# 1. Instalación de Dependencias (IE2)
Instala las librerías principales con versiones fijadas para evitar conflictos en Google Colab. Garantiza la estabilidad del entorno y la compatibilidad con Python 3.10.

In [ ]:
import sys


if "google.colab" in sys.modules:
    !pip install -q "numpy<2.3.0" "pandas<3.0.0" langchain-groq langgraph sentence-transformers python-dotenv pydantic scikit-learn

In [ ]:
!pip install -q groq langchain-text-splitters sentence-transformers numpy python-dotenv

# 2. Configuración de Modelos LLM y API Groq
Carga la credencial de Groq e inicializa el modelo razonador `openai/gpt-oss-120b` para la planificación y orquestación general del agente, junto al modelo `llama-3.2-11b-vision-preview` para análisis multimodal de imágenes.

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Recoger API Key desde los Secretos de Colab
GROQ_API_KEY = userdata.get("LLM_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Asegúrate de agregar 'LLM_API_KEY' en el panel de secretos de Colab.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# LLM Principal para razonamiento y orquestación con openai/gpt-oss-120b
llm_text = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.0)

# LLM Multimodal para lectura de imágenes y recetas médicas
llm_vision = ChatGroq(model_name="llama-3.2-11b-vision-preview", temperature=0.0)

print("✅ Modelo 'openai/gpt-oss-120b' e infraestructura configurados correctamente.")

✅ Modelo 'openai/gpt-oss-120b' e infraestructura configurados correctamente.


# 3. Clasificador de Riesgo ML y Base de Datos SQL
Entrena un modelo Random Forest para detectar pagos sospechosos según hora y monto, e inicializa tablas relacionales en SQLite. Garantiza el cumplimiento de la Ley 19.628 mediante la anonimización de usuarios con hashes.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)

transacciones_legitimas = [{'monto': np.random.uniform(1000, 50000), 'hora': np.random.randint(8, 20), 'riesgo': 0} for _ in range(300)]
transacciones_sospechosas = [{'monto': np.random.uniform(150000, 900000), 'hora': np.random.randint(1, 5), 'riesgo': 1} for _ in range(50)]

df_entrenamiento = pd.DataFrame(transacciones_legitimas + transacciones_sospechosas)
X_train = df_entrenamiento[['monto', 'hora']]
y_train = df_entrenamiento['riesgo']


modelo_ml_riesgo = RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42)
modelo_ml_riesgo.fit(X_train, y_train)


conexion_sql = sqlite3.connect(':memory:', check_same_thread=False)
cursor = conexion_sql.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS transacciones_aprobadas (
        id_tx TEXT, monto REAL, hora INTEGER, usuario_anonimizado TEXT
    )
''')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS transacciones_cuarentena (
        id_tx TEXT, monto REAL, hora INTEGER, usuario_anonimizado TEXT, motivo TEXT
    )
''')
conexion_sql.commit()

print("✅ Modelo Random Forest entrenado y tablas SQL inicializadas.")

✅ Modelo Random Forest entrenado y tablas SQL inicializadas.


# 4. Motor RAG para Búsqueda Semántica de Inventario (IE4)
Indexa los datos de stock, precios y ubicaciones utilizando `SentenceTransformers` y fragmentación de texto. Permite recuperar información precisa mediante búsqueda semántica basada en similitud de coseno.

In [ ]:
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

TEXTO_INVENTARIO = """
INVENTARIO CENTRAL DE FARMACIA:
1. Paracetamol 500mg comprimidos: 150 unidades disponibles. Ubicación: Estante A-1. Precio: $1.990 CLP. Requiere receta: No.
2. Ibuprofeno 400mg cápsulas: 80 unidades disponibles. Ubicación: Estante A-2. Precio: $2.490 CLP. Requiere receta: No.
3. Amoxicilina 500mg: 12 unidades disponibles. Ubicación: Estante B-1. Precio: $4.500 CLP. Requiere receta: Sí (Retenida).
4. Losartán 50mg: 0 unidades disponibles (AGOTADO). Ubicación: Estante C-1. Precio: $6.990 CLP. Requiere receta: Sí.
5. Ketorolaco 10mg: 3 unidades disponibles. Ubicación: Estante D-3. Precio: $3.200 CLP. Requiere receta: Sí (Retenida).
"""

class MotorRAGInventario:
    def __init__(self):
        self.encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.fragmentos: List[str] = []
        self.embeddings: List[np.ndarray] = []

    def indexar(self, texto: str):
        splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
        self.fragmentos = splitter.split_text(texto)
        self.embeddings = self.encoder.encode(self.fragmentos, convert_to_numpy=True)
        print(f"✅ Base RAG indexada con {len(self.fragmentos)} fragmentos semánticos.")

    def recuperar(self, consulta: str, top_k: int = 2) -> str:
        query_vec = self.encoder.encode(consulta, convert_to_numpy=True)
        similitudes = [
            (np.dot(query_vec, doc_vec) / (np.linalg.norm(query_vec) * np.linalg.norm(doc_vec)), frag)
            for doc_vec, frag in zip(self.embeddings, self.fragmentos)
        ]
        similitudes.sort(key=lambda x: x[0], reverse=True)
        return "\n---\n".join([frag for _, frag in similitudes[:top_k]])

motor_rag = MotorRAGInventario()
motor_rag.indexar(TEXTO_INVENTARIO)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Base RAG indexada con 5 fragmentos semánticos.


# 5. Módulo de Planificación Plan-and-Execute
Implementa una cadena de razonamiento que descompone las solicitudes del usuario en una secuencia ordenada de pasos lógicos. Guía el comportamiento autónomo del agente antes de ejecutar cualquier herramienta.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Cadena de planificación basada en la arquitectura Plan-and-Execute[cite: 3]
planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un planificador de operaciones de farmacia. Toma la solicitud del cliente y descompónla en una lista de pasos numéricos simples y ejecutables. Responde únicamente con la lista de pasos."),
    ("human", "Objetivo: {goal}")
])

planner_chain = planner_prompt | llm_text

def generar_plan_trabajo(objetivo: str) -> str:
    res = planner_chain.invoke({"goal": objetivo})
    return res.content

# Verificación de prueba
plan_prueba = generar_plan_trabajo("Revisar stock de Ibuprofeno y validar un pago de $400.000 a las 02:00 AM.")
print("--- PLAN GENERADO ---")
print(plan_prueba)

--- PLAN GENERADO ---
1. Acceder al sistema de gestión de inventario de la farmacia.  
2. Buscar el artículo “Ibuprofeno” utilizando su código o nombre.  
3. Verificar la cantidad disponible en stock y registrar el nivel actual.  
4. Comparar el stock con el umbral mínimo establecido; si está por debajo, generar una orden de reposición.  
5. Acceder al módulo de pagos o facturación del sistema.  
6. Localizar la transacción pendiente de $400.000 programada para las 02:00 AM.  
7. Confirmar que los datos del cliente y la referencia de la factura coinciden con la transacción.  
8. Verificar que el método de pago (tarjeta, transferencia, etc.) esté autorizado y que no existan alertas de fraude.  
9. Aprobar la transacción y registrar la validación en el sistema.  
10. Generar y enviar el comprobante de pago al cliente por correo electrónico o mensaje de texto.  
11. Actualizar el historial de pagos y cerrar la tarea en la lista de pendientes.  
12. Documentar cualquier observación o incid

# 6. Declaración de Herramientas Autónomas
Registra tres herramientas decoradas con `@tool` especializadas en consulta de inventario (RAG), evaluación e inserción SQL de pagos (ML) y análisis OCR/visión de recetas médicas en imagen.

In [ ]:
import base64
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

def crear_receta_prueba(ruta="receta_ejemplo.jpg"):
    img = Image.new('RGB', (400, 150), color=(255, 255, 255))
    d = ImageDraw.Draw(img)
    d.text((10, 10), "RECETA MEDICA", fill=(0, 0, 0))
    d.text((10, 40), "Rx: Amoxicilina 500mg - 1 caja", fill=(0, 0, 0))
    d.text((10, 70), "Dr. Silva - Reg: 45011", fill=(0, 0, 0))
    img.save(ruta)

crear_receta_prueba()

@tool
def consultar_inventario(consulta: str) -> str:
    """Consulta el inventario de la farmacia mediante busqueda semantica RAG para verificar precios, ubicacion y stock."""
    return motor_rag.recuperar(consulta)

@tool
def evaluar_y_registrar_transaccion(monto: float, hora: int) -> str:
    """Evalua el riesgo financiero de una transaccion con Machine Learning (Random Forest) y la almacena en la BD SQL.
    monto: valor numerico del pago en CLP.
    hora: numero entero entre 0 y 23.
    """
    try:
        monto_num = float(monto)
        hora_num = int(hora) % 24
    except Exception:
        monto_num = float(monto) if isinstance(monto, (int, float)) else 0.0
        hora_num = 12

    X_nuevo = pd.DataFrame([{'monto': monto_num, 'hora': hora_num}])
    prediccion_riesgo = modelo_ml_riesgo.predict(X_nuevo)[0]

    hash_usuario = f"USR_HASH_{np.random.randint(10000, 99999)}"
    id_tx = f"TX_{np.random.randint(100, 999)}"

    if prediccion_riesgo == 1:
        cursor.execute(
            "INSERT INTO transacciones_cuarentena VALUES (?, ?, ?, ?, ?)",
            (id_tx, monto_num, hora_num, hash_usuario, "Riesgo detectado por hora/monto")
        )
        conexion_sql.commit()
        return f"Transaccion {id_tx} bloqueada por modelo ML. Guardada en 'transacciones_cuarentena'. Usuario: {hash_usuario}."
    else:
        cursor.execute(
            "INSERT INTO transacciones_aprobadas VALUES (?, ?, ?, ?)",
            (id_tx, monto_num, hora_num, hash_usuario)
        )
        conexion_sql.commit()
        return f"Transaccion {id_tx} aprobada exitosamente. Guardada en 'transacciones_aprobadas'. Usuario: {hash_usuario}."

@tool
def analizar_receta(ruta_imagen: str = "receta_ejemplo.jpg") -> str:
    """Analiza una imagen local de una receta medica para extraer los medicamentos prescriptos."""
    if not os.path.exists(ruta_imagen):
        return f"Error: La imagen '{ruta_imagen}' no existe."

    with open(ruta_imagen, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')

    mensaje = HumanMessage(
        content=[
            {"type": "text", "text": "Identifica los medicamentos descritos en esta receta medica."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
        ]
    )
    res = llm_vision.invoke([mensaje])
    return res.content

lista_herramientas = [
    consultar_inventario,
    evaluar_y_registrar_transaccion,
    analizar_receta
]

print("Herramientas registradas correctamente.")

Herramientas registradas correctamente.


# 7. Orquestación con LangGraph y Memoria
Compila el grafo cíclico de estados en LangGraph conectando el LLM y las herramientas mediante evaluadores de condición. Integra `MemorySaver` para mantener el contexto y seguimiento conversacional por `thread_id`.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# Estado conversacional con memoria de corto plazo acumulativa
class EstadoAgente(TypedDict):
    messages: Annotated[list, add_messages]

builder = StateGraph(EstadoAgente)

# Vinculación del LLM openai/gpt-oss-120b con la lista de herramientas
llm_con_herramientas = llm_text.bind_tools(lista_herramientas)

def nodo_agente(state: EstadoAgente):
    respuesta = llm_con_herramientas.invoke(state["messages"])
    return {"messages": [respuesta]}

# Nodos y bordes del grafo
builder.add_node("agente", nodo_agente)
builder.add_node("tools", ToolNode(lista_herramientas))

builder.add_edge(START, "agente")
builder.add_conditional_edges("agente", tools_condition)
builder.add_edge("tools", "agente")

# Guardado en memoria
memoria_checkpointer = MemorySaver()
agente_orquestador = builder.compile(checkpointer=memoria_checkpointer)

print("✅ Grafo LangGraph compilado con el modelo 'openai/gpt-oss-120b'.")

✅ Grafo LangGraph compilado con el modelo 'openai/gpt-oss-120b'.


# 8. Interfaz Interactiva y Auditoría SQL
Proporciona un menú interactivo en consola para ingresar datos por formulario guiado o texto libre. Ejecuta el agente con streaming de nodos y finaliza auditando las tablas SQL en tiempo real.

In [ ]:
import pandas as pd
import uuid
from datetime import datetime
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Markdown, display

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un planificador operativo de farmacia.
Toma la solicitud del cliente y genera un plan conciso de maximo 3 pasos enfocandote unicamente en las herramientas disponibles:
- Busqueda de inventario RAG (stock, precio, ubicacion y calculo del total a pagar).
- Evaluacion de pagos con ML e insercion en base de datos SQL.
- Analisis OCR de recetas medicas con vision artificial.

Responde unicamente con la lista numerada de pasos sin incluir acciones externas."""),
    ("human", "Objetivo: {goal}")
])

planner_chain = planner_prompt | llm_text

def generar_plan_trabajo(objetivo: str) -> str:
    res = planner_chain.invoke({"goal": objetivo})
    return res.content

def ejecutar_flujo_agente(instruccion: str, thread_id: str):
    plan = generar_plan_trabajo(instruccion)
    print(f"\nPLANIFICACION OPERATIVA DEDUCIDA:\n{plan}\n")

    config = {"configurable": {"thread_id": thread_id}}

    prompt = f"""
    Ejecuta la solicitud del cliente invocando las herramientas correspondientes:

    PLAN DE TRABAJO:
    {plan}

    SOLICITUD DEL CLIENTE:
    {instruccion}

    INSTRUCCIONES ESTRUCTURALES DE EJECUCION:
    1. Para consultar medicamentos, invoca la herramienta 'consultar_inventario'.
    2. Si se debe evaluar o registrar la compra, obtiene el precio unitario del inventario, calcula el precio total multiplicando la cantidad de unidades por ese precio unitario, e invoca 'evaluar_y_registrar_transaccion' usando ese monto total exacto y la hora indicada (entero entre 0 y 23).
    3. Para procesar una receta, invoca 'analizar_receta'.
    4. Responde de forma clara detallando disponibilidad, precio unitario, cantidad solicitada, precio total calculado y el resultado del registro del pago.
    """

    estado_inicial = {"messages": [HumanMessage(content=prompt)]}

    for evento in agente_orquestador.stream(estado_inicial, config):
        for nodo_activo, contenido in evento.items():
            print(f"[Ejecutando Nodo: {nodo_activo}]")

    estado_final = agente_orquestador.get_state(config)
    respuesta_final = estado_final.values["messages"][-1].content

    display(Markdown(f"### Respuesta del Agente:\n{respuesta_final}"))

while True:
    print("\nSISTEMA INTERACTIVO DE FARMACIA Y EVALUACION DE PAGOS")

    modo = input("\nSeleccione una opcion:\n1. Formulario de compra paso a paso\n2. Consulta libre en chat\n3. Salir\n\nOpcion (1, 2 o 3): ").strip()

    if modo == "3" or modo.lower() == "salir":
        print("\nSesion finalizada.")
        break

    if modo == "1":
        medicamento = input("\nNombre del medicamento: ").strip()
        cantidad = input("Cantidad de unidades: ").strip()
        evaluar_pago = input("Desea procesar el pago de esta compra? (s/n): ").strip().lower()

        ahora = datetime.now()
        hora_actual = ahora.hour

        partes = []
        cant_val = cantidad if cantidad else "1"
        if medicamento:
            partes.append(f"Consulta disponibilidad, precio unitario y ubicacion de {cant_val} unidad(es) de {medicamento}.")

        if evaluar_pago == "s":
            partes.append(f"Obten el precio unitario desde el inventario, calcula el precio total multiplicando {cant_val} unidades por dicho precio, e invoca 'evaluar_y_registrar_transaccion' pasando ese precio total calculado y la hora {hora_actual}.")
        else:
            partes.append(f"Calcula y muestra el precio total multiplicando {cant_val} unidades por el precio unitario del inventario.")

        instruccion_final = " ".join(partes)
        if not instruccion_final:
            print("No ingreso datos validos.")
            continue

        thread_id_sesion = f"sesion_{uuid.uuid4().hex[:8]}"
        print(f"\nProcesando solicitud: '{instruccion_final}'")
        ejecutar_flujo_agente(instruccion=instruccion_final, thread_id=thread_id_sesion)

        print("\nAUDITORIA EN BASE DE DATOS SQL")
        df_aprobadas = pd.read_sql_query("SELECT * FROM transacciones_aprobadas", conexion_sql)
        df_cuarentena = pd.read_sql_query("SELECT * FROM transacciones_cuarentena", conexion_sql)

        print("\nTransacciones Aprobadas:")
        print(df_aprobadas.to_string(index=False) if not df_aprobadas.empty else "Sin registros.")
        print("\nTransacciones en Cuarentena:")
        print(df_cuarentena.to_string(index=False) if not df_cuarentena.empty else "Sin registros.")

    elif modo == "2":
        print("\nMODO CONSULTA LIBRE (Escriba 'volver' para regresar al menu principal)")
        while True:
            consulta = input("\nTu pregunta: ").strip()

            if consulta.lower() in ["volver", "salir", "3"]:
                print("Regresando al menu principal...")
                break

            if not consulta:
                print("Por favor escriba una pregunta.")
                continue

            thread_id_sesion = f"sesion_{uuid.uuid4().hex[:8]}"
            print(f"\nProcesando solicitud: '{consulta}'")
            ejecutar_flujo_agente(instruccion=consulta, thread_id=thread_id_sesion)

            print("\nAUDITORIA EN BASE DE DATOS SQL")
            df_aprobadas = pd.read_sql_query("SELECT * FROM transacciones_aprobadas", conexion_sql)
            df_cuarentena = pd.read_sql_query("SELECT * FROM transacciones_cuarentena", conexion_sql)

            print("\nTransacciones Aprobadas:")
            print(df_aprobadas.to_string(index=False) if not df_aprobadas.empty else "Sin registros.")
            print("\nTransacciones en Cuarentena:")
            print(df_cuarentena.to_string(index=False) if not df_cuarentena.empty else "Sin registros.")

    else:
        print("Opcion no valida.")


SISTEMA INTERACTIVO DE FARMACIA Y EVALUACION DE PAGOS

Seleccione una opcion:
1. Formulario de compra paso a paso
2. Consulta libre en chat
3. Salir

Opcion (1, 2 o 3): 1

Nombre del medicamento: Paracetamol
Cantidad de unidades: 4
Desea procesar el pago de esta compra? (s/n): s

Procesando solicitud: 'Consulta disponibilidad, precio unitario y ubicacion de 4 unidad(es) de Paracetamol. Obten el precio unitario desde el inventario, calcula el precio total multiplicando 4 unidades por dicho precio, e invoca 'evaluar_y_registrar_transaccion' pasando ese precio total calculado y la hora 15.'

PLANIFICACION OPERATIVA DEDUCIDA:
1. Utilizar la herramienta de Búsqueda de inventario RAG para obtener el stock disponible, el precio unitario y la ubicación del Paracetamol.  
2. Calcular el precio total multiplicando el precio unitario obtenido por 4 unidades.  
3. Invocar la función `evaluar_y_registrar_transaccion` pasando como parámetros el precio total calculado y la hora 15.

[Ejecutando Nodo

### Respuesta del Agente:
**Disponibilidad y precios**

- **Medicamento:** Paracetamol 500 mg comprimidos  
- **Stock disponible:** 150 unidades  
- **Ubicación:** Estante A‑1  
- **Precio unitario:** **$1.990 CLP**  

**Cálculo del pedido**

- Cantidad solicitada: **4 unidades**  
- Precio total = 4 × $1.990 CLP = **$7.960 CLP**  

**Resultado de la evaluación y registro**

- La transacción fue evaluada y registrada con éxito: **TX_152**, monto **$7.960 CLP**, hora **15:00**.  

Todo está listo para proceder con la entrega del Paracetamol. ¡Gracias por su compra!


AUDITORIA EN BASE DE DATOS SQL

Transacciones Aprobadas:
 id_tx   monto  hora usuario_anonimizado
TX_237  3000.0     3      USR_HASH_49811
TX_500  9000.0     5      USR_HASH_65479
TX_884  1000.0    15      USR_HASH_97369
TX_951 12450.0    15      USR_HASH_29963
TX_152  7960.0    15      USR_HASH_83477

Transacciones en Cuarentena:
Sin registros.

SISTEMA INTERACTIVO DE FARMACIA Y EVALUACION DE PAGOS

Seleccione una opcion:
1. Formulario de compra paso a paso
2. Consulta libre en chat
3. Salir

Opcion (1, 2 o 3): 2

MODO CONSULTA LIBRE (Escriba 'volver' para regresar al menu principal)

Tu pregunta: cuanto stock tenemos de paracetamol

Procesando solicitud: 'cuanto stock tenemos de paracetamol'

PLANIFICACION OPERATIVA DEDUCIDA:
1. Ejecutar una búsqueda RAG en el inventario especificando “paracetamol”.  
2. Obtener los datos de stock disponible, precio unitario y ubicación del producto.  
3. Presentar la cantidad de unidades en existencia y la información asociada.

[Ejecutando Nodo: a

### Respuesta del Agente:
**Disponibilidad de Paracetamol**

- **Producto:** Paracetamol 500 mg comprimidos  
- **Stock disponible:** **150 unidades**  
- **Ubicación:** Estante **A‑1**  
- **Precio unitario:** **$1.990 CLP** (no requiere receta)

Si necesita realizar una compra o desea más información, háganoslo saber y procederemos con el cálculo del total y el registro de la transacción.


AUDITORIA EN BASE DE DATOS SQL

Transacciones Aprobadas:
 id_tx   monto  hora usuario_anonimizado
TX_237  3000.0     3      USR_HASH_49811
TX_500  9000.0     5      USR_HASH_65479
TX_884  1000.0    15      USR_HASH_97369
TX_951 12450.0    15      USR_HASH_29963
TX_152  7960.0    15      USR_HASH_83477

Transacciones en Cuarentena:
Sin registros.

Tu pregunta: volver
Regresando al menu principal...

SISTEMA INTERACTIVO DE FARMACIA Y EVALUACION DE PAGOS

Seleccione una opcion:
1. Formulario de compra paso a paso
2. Consulta libre en chat
3. Salir

Opcion (1, 2 o 3): 3

Sesion finalizada.
